In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/notebooks/diraamelia/01-online-retail-ii-data-cleaning/sales_products_clean.csv",
                 parse_dates=["invoicedate"])

snapshot = df["invoicedate"].max() + pd.Timedelta(days=1)
print("Tanggal snapshot:", snapshot)

rfm = df.groupby("customer_id").agg(
    recency=("invoicedate", lambda x: (snapshot - x.max()).days),
    frequency=("invoice", "nunique"),
    monetary=("total_price", "sum"),
)
print(rfm.shape)
rfm.describe()

## RFM: segmentasi pelanggan
Pelanggan dinilai dari Recency, Frequency, dan Monetary dengan skor kuantil 1-5,
lalu dikelompokkan menjadi 7 segmen. Champions (22,5% pelanggan) menyumbang
70% pendapatan, sedangkan At Risk (245 pelanggan) bernilai tinggi namun sudah
lama tidak belanja sehingga menjadi prioritas kampanye win-back.

In [ ]:
rfm["r"] = pd.qcut(rfm["recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["f"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["m"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).astype(int)
rfm["rfm_score"] = rfm["r"].astype(str) + rfm["f"].astype(str) + rfm["m"].astype(str)

def segmen(row):
    r, f, m = row["r"], row["f"], row["m"]
    fm = (f + m) / 2
    if r >= 4 and fm >= 4:
        return "Champions"
    if r >= 3 and f >= 4:
        return "Loyal"
    if r >= 4 and f <= 2:
        return "New / Promising"
    if r >= 3:
        return "Potential Loyalist"
    if r <= 2 and fm >= 4:
        return "At Risk"
    if r <= 2 and fm >= 2.5:
        return "Need Attention"
    return "Lost"

rfm["segmen"] = rfm.apply(segmen, axis=1)

ringkas = rfm.groupby("segmen").agg(
    pelanggan=("segmen", "size"),
    recency_rata2=("recency", "mean"),
    frekuensi_rata2=("frequency", "mean"),
    monetary_total=("monetary", "sum"),
    monetary_rata2=("monetary", "mean"),
).round(1)
ringkas["persen_pelanggan"] = (ringkas["pelanggan"] / len(rfm) * 100).round(1)
ringkas["persen_pendapatan"] = (ringkas["monetary_total"] / rfm["monetary"].sum() * 100).round(1)
ringkas.sort_values("monetary_total", ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
urut = ringkas.sort_values("pelanggan", ascending=False).index
sns.countplot(data=rfm, y="segmen", order=urut, ax=ax[0])
ax[0].set_title("Jumlah pelanggan per segmen")
sns.barplot(x=ringkas.loc[urut, "persen_pendapatan"], y=urut, ax=ax[1])
ax[1].set_title("Kontribusi pendapatan per segmen (%)")
plt.tight_layout()
plt.show()

## Cohort: retensi pelanggan
Sekitar 21% pelanggan baru kembali belanja di bulan berikutnya, lalu retensi
turun ke kisaran 8-15% setelah beberapa bulan. Pola musiman terlihat jelas:
pelanggan yang masuk di kuartal 4 cenderung kembali di November tahun berikutnya.
Cohort 2009-12 tidak dibandingkan karena berisi pelanggan lama (data mulai
Desember 2009), dan Desember 2011 dikeluarkan karena datanya tidak lengkap.

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt

d = df[df["invoicedate"] < "2011-12-01"].copy()
d["order_month"] = d["invoicedate"].dt.to_period("M")
d["cohort_month"] = d.groupby("customer_id")["invoicedate"].transform("min").dt.to_period("M")
d["periode"] = (d["order_month"] - d["cohort_month"]).apply(lambda x: x.n)

cohort = d.groupby(["cohort_month", "periode"])["customer_id"].nunique().unstack()
retensi = cohort.divide(cohort[0], axis=0) * 100

plt.figure(figsize=(14, 8))
sns.heatmap(retensi, annot=True, fmt=".0f", cmap="Blues", vmax=60, mask=retensi.isna())
plt.title("Retensi pelanggan per cohort (%)", pad=15)
plt.xlabel("Bulan sejak pembelian pertama")
plt.ylabel("Cohort (bulan pembelian pertama)")
plt.show()

Retensi anjlok dari 100% ke sekitar 21% pada bulan pertama, lalu mendatar di
kisaran 14-22%. Kenaikan setelah bulan ke-12 mencerminkan pelanggan musiman
yang kembali belanja di kuartal 4 tahun berikutnya. Ujung kanan grafik dihitung
dari sedikit cohort sehingga kurang stabil.

In [ ]:
rata = retensi.drop(index=pd.Period("2009-12")).mean()
plt.figure(figsize=(9, 4))
plt.plot(rata.index, rata.values, marker="o")
plt.title("Rata-rata retensi per bulan sejak pembelian pertama (%)")
plt.xlabel("Bulan sejak pembelian pertama")
plt.ylabel("Retensi (%)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import os
os.makedirs("/kaggle/working/tableau", exist_ok=True)
out = "/kaggle/working/tableau/"

df["bulan"] = df["invoicedate"].dt.to_period("M").astype(str)

# 1. Pendapatan bulanan (Desember 2011 ditandai tidak lengkap)
bulanan = df.groupby("bulan").agg(
    pendapatan=("total_price", "sum"),
    pesanan=("invoice", "nunique"),
    pelanggan=("customer_id", "nunique"),
).reset_index()
bulanan["status_data"] = bulanan["bulan"].map(lambda b: "Tidak lengkap" if b == "2011-12" else "Lengkap")
bulanan.to_csv(out + "bulanan.csv", index=False)

# 2. Ringkasan segmen RFM
ringkas.reset_index().to_csv(out + "segmen_rfm.csv", index=False)

# 3. Retensi cohort (format panjang)
ret = retensi.stack().reset_index()
ret.columns = ["cohort", "bulan_ke", "retensi_pct"]
ret["cohort"] = ret["cohort"].astype(str)
ret.to_csv(out + "cohort_retensi.csv", index=False)

# 4. Pendapatan per negara
negara = df.groupby("country").agg(
    pendapatan=("total_price", "sum"),
    pelanggan=("customer_id", "nunique"),
    pesanan=("invoice", "nunique"),
).reset_index().sort_values("pendapatan", ascending=False)
negara.to_csv(out + "negara.csv", index=False)

# 5. Top 20 produk
produk = df.groupby("stockcode").agg(
    produk=("description", "first"),
    unit=("quantity", "sum"),
    pendapatan=("total_price", "sum"),
).reset_index().sort_values("pendapatan", ascending=False).head(20)
produk.to_csv(out + "produk_top20.csv", index=False)

print(os.listdir(out))